# 02. Phase 2 — Flow Matching 학습
돌연변이 조건부 Transformer velocity network 학습 (T4 기준 4-6시간)

In [ ]:
import os
os.chdir('/content/brain_idp_flow')

import torch
import yaml
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')

In [ ]:
# ESM-2 임베딩 사전 계산 (1회만)
from brain_idp_flow.targets import load_targets
from brain_idp_flow.features.seq_embed import ESM2Embedder

targets = load_targets('configs/targets.yaml')
embedder = ESM2Embedder(device=device)

seq_embeddings = {}
sid = 0
for tid, target in targets.items():
    # WT
    emb = embedder.embed_single(target.sequence)
    seq_embeddings[sid] = emb.cpu()
    print(f'{tid} WT: seq_id={sid}, emb={emb.shape}')
    sid += 1
    # Mutants
    for mut in target.mutations:
        mut_seq = target.mutant_sequence(mut)
        emb = embedder.embed_single(mut_seq)
        seq_embeddings[sid] = emb.cpu()
        print(f'  {mut.id}: seq_id={sid}, emb={emb.shape}')
        sid += 1

# 캐시 저장
torch.save(seq_embeddings, 'data/seq_embeddings.pt')
print(f'\nTotal: {len(seq_embeddings)} embeddings saved')

In [ ]:
# 학습 실행
from brain_idp_flow.train import train

with open('configs/flow.yaml') as f:
    config = yaml.safe_load(f)

# 캐시에서 임베딩 로드
seq_embeddings = torch.load('data/seq_embeddings.pt', weights_only=True)

# 학습 시작 (T4 기준 ~4-6시간 for 50k steps)
best_ckpt = train(config, seq_embeddings, device)
print(f'\nBest checkpoint: {best_ckpt}')

In [ ]:
# TensorBoard 확인 (선택)
%load_ext tensorboard
%tensorboard --logdir runs/flow

In [ ]:
# 체크포인트를 Google Drive에 백업 (Colab 세션 종료 대비)
from google.colab import drive
drive.mount('/content/drive')

import shutil, glob
ckpt_dir = sorted(glob.glob('runs/flow/*'))[-1]  # latest
backup_dir = '/content/drive/MyDrive/brain_idp_flow_ckpts'
os.makedirs(backup_dir, exist_ok=True)
for f in ['ckpt_best.pt', 'ckpt_last.pt']:
    src = os.path.join(ckpt_dir, f)
    if os.path.exists(src):
        shutil.copy2(src, backup_dir)
        print(f'Backed up {f} -> {backup_dir}')